# **Phase 1 - Silver Layer(Cleaning data and transformations)**

**Read Bronze Tables**

In [0]:
df_customers = spark.table("workspace.default.bronze_customers")

df_devices = spark.table("workspace.default.bronze_devices")

df_jobs = spark.table("workspace.default.bronze_service_jobs")

In [0]:
display(df_jobs)

job_id,customer_id,device_id,issue_type,job_status,received_date,promised_date,completed_date,technician_id,technician_name,repair_notes,estimated_cost,actual_cost
JOB00030,CUST0055,DEV036,Battery Issue,Pending,2024-02-13,2024-02-18,null,T005,Kavitha Nair,null,7662.33,null
JOB01498,CUST0145,DEV037,Motherboard Failure,Completed,2024-02-06,2024-02-11,2024-02-13,T008,Arjun Iyer,null,7751.72,4258.76
JOB01443,CUST0046,DEV027,Motherboard Failure,Completed,2024-03-01,2024-03-06,2024-03-04,T002,Suresh Rao,Customer informed,6912.68,1558.22
JOB00945,CUST0161,DEV018,Water Damage,Completed,2024-03-03,2024-03-08,2024-03-06,T001,Rajesh Kumar,Under warranty,605.69,2263.41
JOB00957,CUST0129,DEV019,RAM Issue,Completed,2024-01-09,2024-01-14,2024-01-10,T003,Priya Singh,Customer informed,5034.79,7386.68
JOB01455,CUST0192,DEV018,Keyboard Fault,Pending,2024-02-19,2024-02-24,null,T008,Arjun Iyer,Customer follow-up pending,4095.16,null
JOB00104,CUST0298,DEV035,Motherboard Failure,Completed,2024-01-18,2024-01-23,2024-01-24,T004,Amit Patel,null,7294.59,3956.07
JOB00294,CUST0022,DEV038,Charging Port Fault,Completed,2024-01-22,2024-01-27,2024-01-26,T005,Kavitha Nair,Under warranty,899.25,6435.76
JOB01400,CUST0036,DEV030,Hinge Broken,Cancelled,2024-01-28,2024-02-02,2024-01-31,T005,Kavitha Nair,null,2197.06,null
JOB01090,CUST0201,DEV017,Overheating,Completed,2024-03-04,2024-03-09,2024-03-10,T003,Priya Singh,Returned to customer,3706.98,7369.66


**Check the Schema**

In [0]:
df_customers.printSchema()

df_devices.printSchema()

df_jobs.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)

root
 |-- device_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- model_series: string (nullable = true)
 |-- warranty_months: integer (nullable = true)
 |-- price_range: string (nullable = true)

root
 |-- job_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- job_status: string (nullable = true)
 |-- received_date: date (nullable = true)
 |-- promised_date: date (nullable = true)
 |-- completed_date: date (nullable = true)
 |-- technician_id: string (nullable = true)
 |-- technician_name: string (nullable = true)
 |-- repair_notes: string (nullable = true)
 |-- 

**Remove Duplicate Records**

In [0]:
df_jobs = df_jobs.dropDuplicates(["job_id"])

**Check Missing Values**

In [0]:
from pyspark.sql.functions import col, sum

df_jobs.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_jobs.columns
]).show()

+------+-----------+---------+----------+----------+-------------+-------------+--------------+-------------+---------------+------------+--------------+-----------+
|job_id|customer_id|device_id|issue_type|job_status|received_date|promised_date|completed_date|technician_id|technician_name|repair_notes|estimated_cost|actual_cost|
+------+-----------+---------+----------+----------+-------------+-------------+--------------+-------------+---------------+------------+--------------+-----------+
|     0|          0|        0|         0|         0|            0|            0|           370|            0|             75|         481|             0|        446|
+------+-----------+---------+----------+----------+-------------+-------------+--------------+-------------+---------------+------------+--------------+-----------+



**Handle Missing Values**

In [0]:
from pyspark.sql.functions import lit

df_jobs = df_jobs.fillna({
    "technician_name": "Not Assigned"
})

**Convert Date Columns**

In [0]:

from pyspark.sql.functions import to_date

df_jobs = df_jobs.withColumn(
    "received_date",
    to_date("received_date")
)

df_jobs = df_jobs.withColumn(
    "promised_date",
    to_date("promised_date")
)

df_jobs = df_jobs.withColumn(
    "completed_date",
    to_date("completed_date")
)

In [0]:
df_jobs.printSchema()

root
 |-- job_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- job_status: string (nullable = true)
 |-- received_date: date (nullable = true)
 |-- promised_date: date (nullable = true)
 |-- completed_date: date (nullable = true)
 |-- technician_id: string (nullable = true)
 |-- technician_name: string (nullable = false)
 |-- repair_notes: string (nullable = true)
 |-- estimated_cost: double (nullable = true)
 |-- actual_cost: double (nullable = true)



**Calculate Repair Duration**

In [0]:
from pyspark.sql.functions import datediff

df_jobs = df_jobs.withColumn(
    "repair_duration",
    datediff("completed_date", "received_date")
)

In [0]:
df_jobs.select(
    "job_id",
    "repair_duration"
).show(10)

+--------+---------------+
|  job_id|repair_duration|
+--------+---------------+
|JOB00030|           NULL|
|JOB01498|              7|
|JOB01443|              3|
|JOB00945|              3|
|JOB00957|              1|
|JOB01455|           NULL|
|JOB00104|              6|
|JOB00294|              4|
|JOB01400|              3|
|JOB01090|              6|
+--------+---------------+
only showing top 10 rows


**Create Delivery Status**

In [0]:
from pyspark.sql.functions import when, col

df_jobs = df_jobs.withColumn(
    "delivery_status",
    when(
        col("completed_date") > col("promised_date"),
        "Delayed"
    ).otherwise("On Time")
)

In [0]:
    df_jobs.select(
    "job_id",
    "delivery_status"
).show(5)

+--------+---------------+
|  job_id|delivery_status|
+--------+---------------+
|JOB00030|        On Time|
|JOB01498|        Delayed|
|JOB01443|        On Time|
|JOB00945|        On Time|
|JOB00957|        On Time|
+--------+---------------+
only showing top 5 rows


**Join the Three Tables**

In [0]:

silver_df = df_jobs.join(
    df_customers,
    on="customer_id",
    how="left"
)

silver_df = silver_df.join(
    df_devices,
    on="device_id",
    how="left"
)

In [0]:
display(silver_df)

device_id,customer_id,job_id,issue_type,job_status,received_date,promised_date,completed_date,technician_id,technician_name,repair_notes,estimated_cost,actual_cost,repair_duration,delivery_status,customer_name,phone_number,email,city,registration_date,brand,device_type,model_series,warranty_months,price_range
DEV019,CUST0295,JOB01181,Battery Issue,Completed,2024-01-29,2024-02-03,2024-01-31,T005,Kavitha Nair,Customer informed,4775.15,6845.21,2,On Time,Meera Khan,9995866934,meera.khan51@outlook.com,Pune,2023-04-09,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV035,CUST0108,JOB00070,Keyboard Fault,Completed,2024-02-29,2024-03-05,2024-03-07,T006,Not Assigned,null,1450.68,4850.57,7,Delayed,Rahul Bajaj,9423573322,rahul.bajaj10@outlook.com,Bhopal,2022-04-14,Motorola,Laptop,Motorola LAP-Series,24,Mid-range (15K-40K)
DEV020,CUST0001,JOB00991,Software Crash,Cancelled,2024-03-13,2024-03-18,2024-03-17,T007,Meena Reddy,Spare part ordered,617.6,null,4,On Time,Kiran Patel,9433218196,kiran.patel5@gmail.com,Delhi,2022-08-12,Lenovo,Smartphone,Lenovo SMA-Series,12,Budget (< 15K)
DEV022,CUST0273,JOB01238,Hard Disk Failure,Pending,2024-01-23,2024-01-28,null,T008,Arjun Iyer,Returned to customer,3234.8,null,null,On Time,Shivam Singh,9074329271,shivam.singh17@hotmail.com,Lucknow,2023-03-16,Asus,Refrigerator,Asus REF-Series,18,Budget (< 15K)
DEV039,CUST0152,JOB00956,Overheating,Completed,2024-01-28,2024-02-02,2024-01-31,T005,Kavitha Nair,Returned to customer,5936.46,4845.37,3,On Time,Nisha Gupta,9874315274,nisha.gupta92@yahoo.com,Mumbai,2022-12-12,Realme,Laptop,Realme LAP-Series,6,Premium (> 40K)
DEV033,CUST0276,JOB00508,Charging Port Fault,In Progress,2024-02-05,2024-02-10,null,T004,Not Assigned,Customer follow-up pending,7955.21,null,null,On Time,Vinay Dubey,9789856257,vinay.dubey46@yahoo.com,Bengaluru,2023-02-18,LG,Smartphone,LG SMA-Series,6,Budget (< 15K)
DEV019,CUST0034,JOB00125,Software Crash,Completed,2024-01-16,2024-01-21,2024-01-19,T003,Priya Singh,null,1836.79,2000.1,3,On Time,Imran Verma,9872148951,imran.verma97@yahoo.com,Lucknow,2022-11-14,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV031,CUST0142,JOB01130,Overheating,Completed,2024-01-21,2024-01-26,2024-01-23,T005,Kavitha Nair,Out of warranty,5232.72,3553.55,2,On Time,Yash Pandey,9705516940,yash.pandey77@hotmail.com,Pune,2023-11-03,LG,Laptop,LG LAP-Series,18,Budget (< 15K)
DEV021,CUST0232,JOB00303,Hard Disk Failure,In Progress,2024-01-28,2024-02-02,null,T003,Priya Singh,Customer follow-up pending,2720.73,null,null,On Time,Anjali Iyer,9490352135,anjali.iyer51@gmail.com,Lucknow,2022-11-03,Lenovo,Monitor,Lenovo MON-Series,18,Mid-range (15K-40K)
DEV043,CUST0181,JOB01475,Motherboard Failure,Completed,2024-01-31,2024-02-05,2024-02-05,T004,Amit Patel,Under warranty,5191.51,6972.09,5,On Time,Deepika Malhotra,9999650477,deepika.malhotra30@outlook.com,Kolkata,2023-01-20,Oppo,Microwave,Oppo MIC-Series,6,Mid-range (15K-40K)


**Data Quality Report**

In [0]:
print("Total Records :", silver_df.count())

print("Total Columns :", len(silver_df.columns))

print("Duplicate Jobs Removed")

print("Date Columns Converted")

print("Delivery Status Created")

Total Records : 1500
Total Columns : 25
Duplicate Jobs Removed
Date Columns Converted
Delivery Status Created


**Save the Silver Table**

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_service_track")

In [0]:
display(
    spark.table("workspace.default.silver_service_track")
)

device_id,customer_id,job_id,issue_type,job_status,received_date,promised_date,completed_date,technician_id,technician_name,repair_notes,estimated_cost,actual_cost,repair_duration,delivery_status,customer_name,phone_number,email,city,registration_date,brand,device_type,model_series,warranty_months,price_range
DEV019,CUST0295,JOB01181,Battery Issue,Completed,2024-01-29,2024-02-03,2024-01-31,T005,Kavitha Nair,Customer informed,4775.15,6845.21,2,On Time,Meera Khan,9995866934,meera.khan51@outlook.com,Pune,2023-04-09,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV035,CUST0108,JOB00070,Keyboard Fault,Completed,2024-02-29,2024-03-05,2024-03-07,T006,Not Assigned,null,1450.68,4850.57,7,Delayed,Rahul Bajaj,9423573322,rahul.bajaj10@outlook.com,Bhopal,2022-04-14,Motorola,Laptop,Motorola LAP-Series,24,Mid-range (15K-40K)
DEV020,CUST0001,JOB00991,Software Crash,Cancelled,2024-03-13,2024-03-18,2024-03-17,T007,Meena Reddy,Spare part ordered,617.6,null,4,On Time,Kiran Patel,9433218196,kiran.patel5@gmail.com,Delhi,2022-08-12,Lenovo,Smartphone,Lenovo SMA-Series,12,Budget (< 15K)
DEV022,CUST0273,JOB01238,Hard Disk Failure,Pending,2024-01-23,2024-01-28,null,T008,Arjun Iyer,Returned to customer,3234.8,null,null,On Time,Shivam Singh,9074329271,shivam.singh17@hotmail.com,Lucknow,2023-03-16,Asus,Refrigerator,Asus REF-Series,18,Budget (< 15K)
DEV039,CUST0152,JOB00956,Overheating,Completed,2024-01-28,2024-02-02,2024-01-31,T005,Kavitha Nair,Returned to customer,5936.46,4845.37,3,On Time,Nisha Gupta,9874315274,nisha.gupta92@yahoo.com,Mumbai,2022-12-12,Realme,Laptop,Realme LAP-Series,6,Premium (> 40K)
DEV033,CUST0276,JOB00508,Charging Port Fault,In Progress,2024-02-05,2024-02-10,null,T004,Not Assigned,Customer follow-up pending,7955.21,null,null,On Time,Vinay Dubey,9789856257,vinay.dubey46@yahoo.com,Bengaluru,2023-02-18,LG,Smartphone,LG SMA-Series,6,Budget (< 15K)
DEV019,CUST0034,JOB00125,Software Crash,Completed,2024-01-16,2024-01-21,2024-01-19,T003,Priya Singh,null,1836.79,2000.1,3,On Time,Imran Verma,9872148951,imran.verma97@yahoo.com,Lucknow,2022-11-14,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV031,CUST0142,JOB01130,Overheating,Completed,2024-01-21,2024-01-26,2024-01-23,T005,Kavitha Nair,Out of warranty,5232.72,3553.55,2,On Time,Yash Pandey,9705516940,yash.pandey77@hotmail.com,Pune,2023-11-03,LG,Laptop,LG LAP-Series,18,Budget (< 15K)
DEV021,CUST0232,JOB00303,Hard Disk Failure,In Progress,2024-01-28,2024-02-02,null,T003,Priya Singh,Customer follow-up pending,2720.73,null,null,On Time,Anjali Iyer,9490352135,anjali.iyer51@gmail.com,Lucknow,2022-11-03,Lenovo,Monitor,Lenovo MON-Series,18,Mid-range (15K-40K)
DEV043,CUST0181,JOB01475,Motherboard Failure,Completed,2024-01-31,2024-02-05,2024-02-05,T004,Amit Patel,Under warranty,5191.51,6972.09,5,On Time,Deepika Malhotra,9999650477,deepika.malhotra30@outlook.com,Kolkata,2023-01-20,Oppo,Microwave,Oppo MIC-Series,6,Mid-range (15K-40K)


Q1. Why we use Silver Layer?
Ans: In this phase, the raw data from the Bronze Layer was cleaned and transformed. Duplicate records were removed, date columns were converted into the correct format, repair duration was calculated, and the delivery status of each job was identified. Finally, the customer, device, and service job datasets were combined into a single table and saved as a Silver Delta table for further analysis.